<a href="https://colab.research.google.com/github/hemanthgm914856/CNN-Model-to-identify-Cat-and-Dog/blob/main/CNN_MODEL_FOR_CAT_DOG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🐱🐶 CAT vs DOG CNN - CLEAN DATASET VERSION
# Google Colab | Single Cell
# ============================================================

import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import kagglehub

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from sklearn.metrics import confusion_matrix, classification_report
from google.colab import files
from PIL import Image

# ============================================================
# 1. CHECK TENSORFLOW AND GPU
# ============================================================

print("TensorFlow Version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# ============================================================
# 2. DOWNLOAD DATASET
# ============================================================

print("\n📥 Downloading Cats vs Dogs dataset...")

dataset_path = kagglehub.dataset_download(
    "shaunthesheep/microsoft-catsvsdogs-dataset"
)

print("Dataset downloaded to:")
print(dataset_path)

# ============================================================
# 3. FIND CAT AND DOG DIRECTORIES
# ============================================================

cat_dir = None
dog_dir = None

for root, dirs, files_list in os.walk(dataset_path):

    folder = os.path.basename(root).lower()

    if folder == "cat":
        cat_dir = root

    elif folder == "dog":
        dog_dir = root

print("\n🐱 Cat folder:", cat_dir)
print("🐶 Dog folder:", dog_dir)

if cat_dir is None or dog_dir is None:
    raise Exception(
        "Could not automatically find Cat and Dog folders."
    )

# ============================================================
# 4. CREATE CLEAN DATASET
# ============================================================

clean_dir = "/content/clean_cat_dog_dataset"

cat_clean = os.path.join(clean_dir, "Cat")
dog_clean = os.path.join(clean_dir, "Dog")

os.makedirs(cat_clean, exist_ok=True)
os.makedirs(dog_clean, exist_ok=True)

print("\n🔍 Checking images for corruption...")

valid_count = 0
invalid_count = 0

def clean_images(source_dir, destination_dir):

    global valid_count, invalid_count

    for filename in os.listdir(source_dir):

        source_file = os.path.join(
            source_dir,
            filename
        )

        destination_file = os.path.join(
            destination_dir,
            filename
        )

        # Skip directories
        if not os.path.isfile(source_file):
            continue

        # Only image files
        if not filename.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".gif")
        ):
            continue

        try:

            # Open image
            img = Image.open(source_file)

            # Verify image integrity
            img.verify()

            # Re-open because verify() closes the file
            img = Image.open(source_file)

            # Convert to RGB
            img = img.convert("RGB")

            # Save clean JPEG
            new_name = os.path.splitext(filename)[0] + ".jpg"

            destination_file = os.path.join(
                destination_dir,
                new_name
            )

            img.save(
                destination_file,
                "JPEG",
                quality=95
            )

            valid_count += 1

        except Exception:

            invalid_count += 1

            print(
                "❌ Removed invalid image:",
                filename
            )


clean_images(
    cat_dir,
    cat_clean
)

clean_images(
    dog_dir,
    dog_clean
)

print("\n==========================================")
print("DATASET CLEANING COMPLETE")
print("==========================================")

print("Valid images  :", valid_count)
print("Invalid images:", invalid_count)

# ============================================================
# 5. LOAD CLEAN DATASET
# ============================================================

IMG_SIZE = (150, 150)
BATCH_SIZE = 32

print("\n📚 Loading clean dataset...")

train_ds = tf.keras.utils.image_dataset_from_directory(

    clean_dir,

    validation_split=0.20,

    subset="training",

    seed=42,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(

    clean_dir,

    validation_split=0.20,

    subset="validation",

    seed=42,

    image_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    label_mode="binary",

    shuffle=False
)

class_names = train_ds.class_names

print("\n🐾 Classes:", class_names)

# ============================================================
# 6. OPTIMIZE DATA PIPELINE
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# ============================================================
# 7. DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.1),

    layers.RandomZoom(0.1),

    layers.RandomContrast(0.1)

])

# ============================================================
# 8. BUILD CNN
# ============================================================

model = models.Sequential([

    layers.Input(
        shape=(150, 150, 3)
    ),

    # Data augmentation
    data_augmentation,

    # Normalize
    layers.Rescaling(1./255),

    # ---------------- CNN BLOCK 1 ----------------

    layers.Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        2,2
    ),

    # ---------------- CNN BLOCK 2 ----------------

    layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        2,2
    ),

    # ---------------- CNN BLOCK 3 ----------------

    layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        2,2
    ),

    # ---------------- CNN BLOCK 4 ----------------

    layers.Conv2D(
        256,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D(
        2,2
    ),

    # ---------------- CLASSIFIER ----------------

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(
        0.5
    ),

    # Binary classification
    layers.Dense(
        1,
        activation="sigmoid"
    )
])

# ============================================================
# 9. COMPILE
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy",

    metrics=["accuracy"]
)

print("\n🧠 CNN MODEL ARCHITECTURE")

model.summary()

# ============================================================
# 10. EARLY STOPPING
# ============================================================

early_stopping = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True
)

# ============================================================
# 11. TRAIN MODEL
# ============================================================

print("\n🚀 TRAINING CNN...\n")

EPOCHS = 15

history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=[
        early_stopping
    ]
)

# ============================================================
# 12. EVALUATE MODEL
# ============================================================

loss, accuracy = model.evaluate(
    val_ds
)

print("\n==========================================")
print("🐱🐶 FINAL MODEL PERFORMANCE")
print("==========================================")

print(
    f"Validation Accuracy : {accuracy*100:.2f}%"
)

print(
    f"Validation Loss     : {loss:.4f}"
)

# ============================================================
# 13. ACCURACY GRAPH
# ============================================================

plt.figure(
    figsize=(8,5)
)

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title(
    "🐱🐶 Cat vs Dog CNN - Accuracy"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)

plt.legend()

plt.grid()

plt.show()

# ============================================================
# 14. LOSS GRAPH
# ============================================================

plt.figure(
    figsize=(8,5)
)

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title(
    "🐱🐶 Cat vs Dog CNN - Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.legend()

plt.grid()

plt.show()

# ============================================================
# 15. CONFUSION MATRIX
# ============================================================

print("\n📊 Creating Confusion Matrix...")

y_true = []
y_pred = []

for images, labels in val_ds:

    predictions = model.predict(
        images,
        verbose=0
    )

    predicted_labels = (
        predictions.flatten() >= 0.5
    ).astype(int)

    y_true.extend(
        labels.numpy().astype(int).flatten()
    )

    y_pred.extend(
        predicted_labels
    )

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(
    figsize=(7,6)
)

sns.heatmap(

    cm,

    annot=True,

    fmt="d",

    cmap="Blues",

    xticklabels=class_names,

    yticklabels=class_names
)

plt.title(
    "🐱🐶 Cat vs Dog - Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)

plt.tight_layout()

plt.show()

# ============================================================
# 16. CLASSIFICATION REPORT
# ============================================================

print("\n📋 CLASSIFICATION REPORT\n")

print(
    classification_report(

        y_true,

        y_pred,

        target_names=class_names
    )
)

# ============================================================
# 17. SAVE MODEL
# ============================================================

model_path = "/content/cat_dog_cnn.keras"

model.save(
    model_path
)

print("\n✅ Model saved successfully:")
print(model_path)

# ============================================================
# 18. TEST YOUR OWN IMAGE
# ============================================================

print("\n==========================================")
print("📷 TEST YOUR OWN IMAGE")
print("==========================================")

uploaded = files.upload()

for filename in uploaded.keys():

    img = image.load_img(

        filename,

        target_size=IMG_SIZE
    )

    img_array = image.img_to_array(
        img
    )

    img_array = np.expand_dims(
        img_array,
        axis=0
    )

    prediction = model.predict(
        img_array,
        verbose=0
    )[0][0]

    # Class 0 = Cat
    # Class 1 = Dog

    if prediction >= 0.5:

        predicted_class = "DOG 🐶"

        confidence = prediction * 100

    else:

        predicted_class = "CAT 🐱"

        confidence = (
            1 - prediction
        ) * 100

    plt.figure(
        figsize=(6,6)
    )

    plt.imshow(
        img
    )

    plt.axis(
        "off"
    )

    plt.title(

        f"Prediction: {predicted_class}\n"
        f"Confidence: {confidence:.2f}%"

    )

    plt.show()

    print("\n==========================================")
    print("🔍 PREDICTION RESULT")
    print("==========================================")

    print(
        "Image      :",
        filename
    )

    print(
        "Prediction :",
        predicted_class
    )

    print(
        f"Confidence : {confidence:.2f}%"
    )

    print("==========================================")

print("\n🎉 CAT vs DOG CNN PROJECT COMPLETED!")